In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

In [2]:
df_bronze = spark.read.table("default.regularite_mensuelle_tgv_aqst")

In [3]:
df_filtered = (df_bronze
             .drop("Commentaire annulations", 
                   "Commentaire retards au départ", 
                   "Commentaire retards à l'arrivée"))

for col_name in df_filtered.columns:
    clean_name = (col_name.lower()
                  .replace(" ", "_")
                  .replace("'", "_")
                  .replace(">", "sup")
                  .replace("(", "")
                  .replace(")", "")
                  .replace("/", "_")
                  .replace("__", "_")
                  .replace("é", "e")
                  .replace("è", "e")
                  .replace("à", "a"))
    df_filtered = df_filtered.withColumnRenamed(col_name, clean_name)

#Toutes les colonnes avec le type long
column_to_cast = [
    "duree_moyenne_du_trajet", "nombre_de_circulations_prevues", "nombre_de_trains_annules",
    "nombre_de_trains_en_retard_au_depart", "nombre_de_trains_en_retard_a_l_arrivee",
    "nombre_trains_en_retard_sup_15min", "nombre_trains_en_retard_sup_30min", "nombre_trains_en_retard_sup_60min"
]

for c in column_to_cast:
    df_filtered = df_filtered.withColumn(c, col(c).cast("int"))


df_silver_clean = df_filtered.na.fill(value = 0, subset = column_to_cast)

df_silver_clean.printSchema()

root
 |-- date: date (nullable = true)
 |-- service: string (nullable = true)
 |-- gare_de_depart: string (nullable = true)
 |-- gare_d_arrivee: string (nullable = true)
 |-- duree_moyenne_du_trajet: integer (nullable = false)
 |-- nombre_de_circulations_prevues: integer (nullable = false)
 |-- nombre_de_trains_annules: integer (nullable = false)
 |-- nombre_de_trains_en_retard_au_depart: integer (nullable = false)
 |-- retard_moyen_des_trains_en_retard_au_depart: double (nullable = true)
 |-- retard_moyen_de_tous_les_trains_au_depart: double (nullable = true)
 |-- nombre_de_trains_en_retard_a_l_arrivee: integer (nullable = false)
 |-- retard_moyen_des_trains_en_retard_a_l_arrivee: double (nullable = true)
 |-- retard_moyen_de_tous_les_trains_a_l_arrivee: double (nullable = true)
 |-- nombre_trains_en_retard_sup_15min: integer (nullable = false)
 |-- retard_moyen_trains_en_retard_sup_15_si_liaison_concurrencee_par_vol: double (nullable = true)
 |-- nombre_trains_en_retard_sup_30min: in

In [ ]:
# On écrit directement la table propre dans le catalogue
(df_silver_clean.write
    .mode("overwrite") # Écrase si elle existe déjà, idéal pour tes tests
    .saveAsTable("default.sncf_silver_clean"))

print("Table Silver enregistrée et synchronisée dans Unity Catalog.")

Table Silver enregistrée et synchronisée dans Unity Catalog.
